In [1]:
# Dataset: IMDB 50K Movie Reviews
# Kaggle: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
#
# Option 1 (Recommended - Auto Download): Run the download cell below
# Option 2 (Manual): Download "IMDB Dataset.csv" from Kaggle and place it
#   in the same folder, then use: df = pd.read_csv("IMDB Dataset.csv")

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Auto-download the IMDB dataset (mirrored on GitHub)
import urllib.request

url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
urllib.request.urlretrieve(url, "IMDB Dataset.csv")
print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [4]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.replace({
    'sentiment': {
        'positive': 1,
        'negative': 0
    }
    }, inplace = True)

/tmp/ipykernel_7193/406208653.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({


In [6]:
df['review'] = df['review'].str.lower()

In [7]:
df.drop_duplicates(inplace = True)

In [8]:
import re
def remove_html_tags(text):
  pattern = r'<.*?>'
  text = re.sub(pattern, ' ', text)
  return text
df['review'] = df['review'].apply(remove_html_tags)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming t...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [9]:
punc_pattern = r'[^a-zA-Z\s]'
df['review'] = df['review'].apply(lambda x: re.sub(punc_pattern, ' ', x))

In [10]:
import nltk

In [11]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df['review'] = df['review'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
def remove_urls(text):
  url = r'https?://\S+|www\.\S+'
  return re.sub(url, ' ', text)
df['review'] = df['review'].apply(remove_urls)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
X = df['review']
y = df['sentiment']

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

In [16]:
tfidf = TfidfVectorizer(max_features = 5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [17]:
logistic_model = LogisticRegression(max_iter=500)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=500)

In [18]:
y_pred = logistic_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of logistic model: {accuracy: .4f}")

Accuracy of logistic model:  0.8886


In [19]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

RandomForestClassifier(random_state=42)

In [20]:
y_pred_rf = rf_model.predict(X_test_tfidf)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy of random forest model: {accuracy_rf: .4f}")

Accuracy of random forest model:  0.8364


In [21]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)

KeyboardInterrupt: 

In [ ]:
y_pred_dt = dt_model.predict(X_test_tfidf)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy of decision tree model: {accuracy_dt: .4f}")

In [ ]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_tfidf)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy of Xgboost model: {accuracy_xgb: .4f}")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred_knn = knn_model.predict(X_test_tfidf)
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy of KNN model: {accuracy_knn: .4f}")

In [ ]:
from sklearn.svm import SVC
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred_svm = svm_model.predict(X_test_tfidf)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy of SVM model: {accuracy_svm: .4f}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.2 ,random_state=42)

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])
X_train_lstm = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test_lstm = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [ ]:
Y_train_lstm = train_data['sentiment']
Y_test_lstm = test_data['sentiment']

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train_lstm, Y_train_lstm, validation_data=(X_test_lstm, Y_test_lstm), epochs=10, batch_size=64)

In [ ]:
pred = model.predict(X_test_lstm)

In [ ]:
y_pred_classes = (pred > 0.5).astype("int32")

In [ ]:
label_map = {1: "Positive", 0: "Negative"}
print("Top 10 Predictions")
for i in range(10):
    actual = label_map[y_test.values[i]]
    predicted = label_map[y_pred_classes[i][0]]
    print(f"Sample {i+1}: Actual: {actual} | Predicted: {predicted}")